# Weight PTQ and QAT gradients

Quantization here uses floating Q/DQ values. This notebook illustrates the grid and verifies that QAT gradients reach master weights.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / "edgeformer").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "edgeformer").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks folder")
sys.path.insert(0, str(ROOT))
torch.set_num_threads(1)
torch.manual_seed(42)
from edgeformer.models import TinyViT

model = TinyViT(
    image_size=16,
    patch_size=4,
    num_classes=4,
    embed_dim=32,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.0,
)

In [ ]:
from edgeformer.quantization import fake_quantize_tensor, quantize_model_weights, prepare_qat

values = torch.linspace(-1, 1, 100)
import matplotlib.pyplot as plt

for bits in [4, 8]:
    plt.step(values.numpy(), fake_quantize_tensor(values, bits).numpy(), label=f"W{bits}")
plt.plot([-1, 1], [-1, 1], "--", label="identity")
plt.xlabel("Floating input")
plt.ylabel("Q/DQ output")
plt.legend()

In [ ]:
images = torch.randn(4, 3, 16, 16)
model.eval()
with torch.inference_mode():
    reference = model(images)
    w4 = quantize_model_weights(model, 4)
    candidate = w4(images)
from edgeformer.analysis import output_distortion

output_distortion(reference, candidate)

In [ ]:
qat = prepare_qat(model, 4).train()
loss = qat(images).square().mean()
loss.backward()
assert qat.head.weight.grad is not None
assert qat.patch_embed.weight.grad.abs().sum() > 0
print("Head gradient norm:", qat.head.weight.grad.norm().item())
print("Patch gradient norm:", qat.patch_embed.weight.grad.norm().item())

The model above is untrained. Distortion is a numerical example, not evidence that W4 preserves classification accuracy. QAT dynamically quantizes Conv/Linear inputs and weights; attention products and nonlinear operations remain floating.